## 1. Setup & Imports
All third-party imports, device selection, and global random seed.

In [ ]:
import os, math, random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from scipy.spatial.distance import pdist, squareform
from sklearn import linear_model
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 2. Configuration
All hyperparameters and paths in one place — change here only.

In [ ]:
cfg = SimpleNamespace(
    # --- Paths ---
    dataset_dir="/kaggle/input/competitions/csiro-biomass",
    dino_weights_dir="/kaggle/input/datasets/darealvictorslorer/dinov2-small-weights/dinov2-small",
    output_dir="/kaggle/working",
    # --- Model ---
    input_dim=384,
    latent_dim=64,
    output_dim=5,
    dropout=0.3,
    # --- CEMS ---
    sigma=1e-3,          # used when sigma_auto=False
    sigma_auto=True,     # True → derive sigma from median NN distance in z-space
    sigma_fraction=0.1,  # sigma = median_nn_dist * sigma_fraction (only when sigma_auto=True)
    sigma_dedup=True,    # True → use one variant per image for NN distance; False → use all augmented samples
    cems_method=1,
    use_hessian=True,
    input_sampling=True,     # CEMS in raw 384-d feature space → full model forward
    manifold_sampling=False, # CEMS in 64-d latent space via forward_cems; mutually exclusive with input_sampling
    mix_real=True,           # True → loss = loss_real + lambda_aug * loss_aug; False → loss_aug only
    lambda_aug=1.0,          # weight on augmented loss term (used when mix_real=True)
    # --- Neighbourhood ---
    neigh_type="knn",        # "knn" or "random"
    # --- Augmentation ---
    # Subset of ["identity", "hflip", "vflip", "hflip_vflip"]; must include "identity"
    augmentations=["identity", "hflip_vflip"],
    # --- Training ---
    epochs=80,
    lr=3e-4,
    weight_decay=1e-3,
    seed=42,
    batch_size=64,
    # --- Split ---
    n_splits=5,
    val_fold=0,
)

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1,
           "GDM_g": 0.2, "Dry_Total_g": 0.5}

TRAIN_CSV     = os.path.join(cfg.dataset_dir, "train.csv")
TEST_CSV      = os.path.join(cfg.dataset_dir, "test.csv")
TRAIN_IMG_DIR = os.path.join(cfg.dataset_dir, "train")
TEST_IMG_DIR  = os.path.join(cfg.dataset_dir, "test")

print("train.csv:",  os.path.exists(TRAIN_CSV))
print("test.csv:",   os.path.exists(TEST_CSV))
print("train dir:",  os.path.exists(TRAIN_IMG_DIR))
print("test dir:",   os.path.exists(TEST_IMG_DIR))
print("dino dir:",   os.path.exists(cfg.dino_weights_dir))

## 3. Load DINOv2
Load ViT-S/14 from uploaded HuggingFace weights; smoke-test at 504×252 resolution.

In [3]:
print(f"Loading DINOv2-small from {cfg.dino_weights_dir} ...")
dino = AutoModel.from_pretrained(cfg.dino_weights_dir).eval().to(device)
for p in dino.parameters():
    p.requires_grad_(False)

# Smoke test: 504x252 input (W=504, H=252) — must pass interpolate_pos_encoding=True
# because this is a non-standard resolution for the HF wrapper.
_dummy = torch.zeros(1, 3, 252, 504, device=device)  # (B, C, H, W)
with torch.no_grad():
    _out = dino(pixel_values=_dummy, interpolate_pos_encoding=True)
    _cls = _out.last_hidden_state[:, 0, :]  # CLS token
assert _cls.shape == (1, 384), f"Expected (1, 384), got {_cls.shape}"
print(f"DINOv2 smoke test passed — CLS shape: {_cls.shape}")
del _dummy, _out, _cls

Loading DINOv2 ViT-S/14 from /kaggle/input/datasets/darealvictorslorer/dinov2-small-weights/dinov2-small ...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 smoke test passed — CLS shape: torch.Size([1, 384])


## 4. Load Training CSV
Pivot long-format CSV to wide format (one row per image) and extract Y matrix.

In [4]:
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["image_id"] = df_raw["sample_id"].str.split("__").str[0]

df_wide = df_raw.pivot_table(
    index=["image_id", "image_path"],
    columns="target_name",
    values="target",
).reset_index()

Y_all = df_wide[TARGETS].values.astype(np.float32)  # (N, 5)
train_image_ids_all = df_wide["image_id"].values

print(f"Training images: {len(df_wide)}")
print(f"Y_all shape:     {Y_all.shape}")
print(df_wide.head(3))

Training images: 357
Y_all shape:     (357, 5)
target_name      image_id              image_path  Dry_Clover_g  Dry_Dead_g  \
0            ID1011485656  train/ID1011485656.jpg          0.00     31.9984   
1            ID1012260530  train/ID1012260530.jpg          0.00      0.0000   
2            ID1025234388  train/ID1025234388.jpg          6.05      0.0000   

target_name  Dry_Green_g  Dry_Total_g   GDM_g  
0                16.2751      48.2735  16.275  
1                 7.6000       7.6000   7.600  
2                 0.0000       6.0500   6.050  


## 5. Extract DINOv2 Features — Training Images
Preprocess at 504×252 and extract CLS tokens with configurable augmentations (identity, hflip, vflip, hflip+vflip).

In [ ]:
# 504x252 = (W x H); torchvision Resize takes (H, W) = (252, 504)
# Matches extract_features.py: DINOV2_RESIZE=(504,252), Resize((252,504))
_dino_transform = T.Compose([
    T.Resize((252, 504)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def extract_features(image_paths, model, transform):
    """Return (N, 384) float32 array of CLS-token features."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(pixel_values=x, interpolate_pos_encoding=True)
            feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
        feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


_AUGMENTATION_FNS = {
    "identity":    lambda img: img,
    "hflip":       lambda img: TF.hflip(img),
    "vflip":       lambda img: TF.vflip(img),
    "hflip_vflip": lambda img: TF.hflip(TF.vflip(img)),
}
_ORIENTATIONS = [_AUGMENTATION_FNS[a] for a in cfg.augmentations]
n_aug = len(_ORIENTATIONS)


def extract_features_augmented(image_paths, model, transform):
    """Return (N*n_aug, 384) array — each image appears n_aug times."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        for flip_fn in _ORIENTATIONS:
            x = transform(flip_fn(img)).unsqueeze(0).to(device)
            with torch.no_grad():
                out = model(pixel_values=x, interpolate_pos_encoding=True)
                feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
            feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


train_image_paths = [
    os.path.join(TRAIN_IMG_DIR, f"{iid}.jpg")
    for iid in train_image_ids_all
]
N_orig = len(train_image_paths)
print(f"Extracting features for {N_orig} training images ({n_aug} orientations each)...")
X_all = extract_features_augmented(train_image_paths, dino, _dino_transform)

# Expand Y and image IDs to match augmented feature array
Y_all               = np.repeat(Y_all,               n_aug, axis=0)
train_image_ids_all = np.repeat(train_image_ids_all, n_aug)
image_group_ids     = np.repeat(np.arange(N_orig),   n_aug)

print(f"X_all shape:         {X_all.shape}")
print(f"Y_all shape:         {Y_all.shape}")
print(f"image_group_ids len: {len(image_group_ids)}")
print(f"Unique groups:       {len(np.unique(image_group_ids))}")

# Sanity checks
assert X_all.shape == (N_orig * n_aug, 384), f"Expected ({N_orig*n_aug}, 384), got {X_all.shape}"
assert Y_all.shape == (N_orig * n_aug, 5)
assert len(image_group_ids) == N_orig * n_aug
assert len(np.unique(image_group_ids)) == N_orig
assert np.all(np.bincount(image_group_ids) == n_aug), "Each group must appear exactly n_aug times"
print("Sanity checks passed.")

Extracting features for 357 training images (2 orientations each)...
  50/357
  100/357
  150/357
  200/357
  250/357
  300/357
  350/357
X_all shape:         (714, 384)
Y_all shape:         (714, 5)
image_group_ids len: 714
Unique groups:       357
Sanity checks passed.


## 6. Extract DINOv2 Features — Test Images
Same preprocessing pipeline applied to the held-out test set.

In [6]:
df_test_raw = pd.read_csv(TEST_CSV)
df_test_raw["image_id"] = df_test_raw["sample_id"].str.split("__").str[0]
df_test_unique = df_test_raw.drop_duplicates("image_id").copy()

test_image_ids = df_test_unique["image_id"].values
test_image_paths = [
    os.path.join(TEST_IMG_DIR, f"{iid}.jpg")
    for iid in test_image_ids
]

print(f"Extracting features for {len(test_image_paths)} test images...")
X_test = extract_features(test_image_paths, dino, _dino_transform)
print(f"X_test shape: {X_test.shape}")

Extracting features for 1 test images...
X_test shape: (1, 384)


## 7. GroupKFold Train/Val Split
Fold 0 of a 5-fold GroupKFold; groups on image_path (one unique group per image).

In [7]:
gkf = GroupKFold(n_splits=cfg.n_splits)
# Group by original image index so all 4 orientations of an image land in the
# same fold — prevents leakage (original in train, hflip in val = cheating).
splits = list(gkf.split(X_all, groups=image_group_ids))
train_idx, val_idx = splits[cfg.val_fold]

X_train     = X_all[train_idx]
Y_train_raw = Y_all[train_idx]
X_val       = X_all[val_idx]
Y_val_raw   = Y_all[val_idx]
train_ids_split = train_image_ids_all[train_idx]
val_ids_split   = train_image_ids_all[val_idx]

# Sanity check: no original image shared across train and val
train_groups = set(image_group_ids[train_idx].tolist())
val_groups   = set(image_group_ids[val_idx].tolist())
assert train_groups.isdisjoint(val_groups), "Group leakage: original images shared across folds!"
print("Group overlap check: PASSED (zero overlap)")

print(f"Train: {X_train.shape}  Val: {X_val.shape}")
print(f"Y_train range: [{Y_train_raw.min():.1f}, {Y_train_raw.max():.1f}]")

Group overlap check: PASSED (zero overlap)
Train: (570, 384)  Val: (144, 384)
Y_train range: [0.0, 185.7]


## 8. MinMaxScaler on Y_train
Fit scaler on training labels only; used inside CEMS joint space for label normalisation.

In [8]:
y_scaler = MinMaxScaler()
y_scaler.fit(Y_train_raw)
Y_train_scaled = y_scaler.transform(Y_train_raw).astype(np.float32)

print(f"Y_train_scaled range: [{Y_train_scaled.min():.3f}, {Y_train_scaled.max():.3f}]")
print("Scaler data_min_:", y_scaler.data_min_.round(2))
print("Scaler data_max_:", y_scaler.data_max_.round(2))

Y_train_scaled range: [0.000, 1.000]
Scaler data_min_: [0.   0.   0.   1.4  2.48]
Scaler data_max_: [157.98  83.84  71.79 157.98 185.7 ]


## 9. Model Architecture — Encoder, Head, BiomassModel
Encoder 384→256→64 (GELU, Dropout), Head 64→32→5 (GELU, no dropout).
The fresh phase-2 head is instantiated as a plain `nn.Sequential` without dropout — CEMS is the regularizer.

In [9]:
class Encoder(nn.Module):
    """384 → 256 → 64 (GELU, Dropout)."""

    def __init__(self, input_dim=384, latent_dim=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class Head(nn.Module):
    """64 → 32 → 5 (GELU, Dropout)."""

    def __init__(self, latent_dim=64, output_dim=5, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class BiomassModel(nn.Module):

    def __init__(self, input_dim=384, latent_dim=64, output_dim=5, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim, dropout)
        self.head = Head(latent_dim, output_dim, dropout)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.head(self.encode(x))

    def forward_cems(self, args, x, y_scaled):
        """CEMS augmentation path.

        Encodes x, detaches latent so SVD/ridge don't backprop through encoder,
        then runs get_batch_cems in 64-d latent space.
        Returns (pred_aug, y_aug_scaled).
        """
        latent = self.encode(x)
        latent_aug, y_aug_scaled = get_batch_cems(
            args, latent.detach(), y_scaled, latent=True,
        )
        return self.head(latent_aug), y_aug_scaled


print("BiomassModel defined.")
# Quick parameter count
_tmp = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout)
n_params = sum(p.numel() for p in _tmp.parameters())
print(f"Parameters: {n_params:,}")
del _tmp

BiomassModel defined.
Parameters: 117,253


## 10. CEMS Algorithm
Port of cems.py: _adjust_dims, _get_projection, _estimate_grad_hessian, _sample_tangent, get_batch_cems.

In [ ]:
# ---------------------------------------------------------------------------
# TwoNN intrinsic dimension estimator (Facco et al. 2017)
# Copied from CEMS-main/src/intrinsic_dimension.py
# ---------------------------------------------------------------------------

def _estimate_np(X, fraction=0.9):
    """TwoNN slope estimator on a precomputed (n, n) distance matrix."""
    Y = np.sort(X, axis=1, kind='quicksort')
    k1 = Y[:, 1]
    k2 = Y[:, 2]

    zeros        = np.where(k1 == 0)[0]
    degeneracies = np.where(k1 == k2)[0]
    good = np.setdiff1d(np.arange(Y.shape[0]), zeros)
    good = np.setdiff1d(good, degeneracies)

    k1 = k1[good]
    k2 = k2[good]

    npoints = int(np.floor(good.shape[0] * fraction))
    N       = good.shape[0]
    mu      = np.sort(np.divide(k2, k1), kind='quicksort')
    Femp    = np.arange(1, N + 1, dtype=np.float64) / N

    x = np.log(mu[:-2])
    y = -np.log(1 - Femp[:-2])

    regr = linear_model.LinearRegression(fit_intercept=False)
    regr.fit(x[:npoints, np.newaxis], y[:npoints, np.newaxis])
    return regr.coef_[0][0]


def twonn_inhouse(X):
    """TwoNN intrinsic dimension estimate (Facco et al. 2017). Returns float."""
    dist = squareform(pdist(X, metric='euclidean'))
    return _estimate_np(dist)


# ---------------------------------------------------------------------------
# Step 1 — flatten + form joint space zi = [x | y]
# ---------------------------------------------------------------------------

def _adjust_dims(x, y, xk=None, yk=None):
    if x.ndim > 2:
        x = x.reshape(x.shape[0], -1)
    if y.ndim == 1:
        y = y.reshape(y.shape[0], 1)
        
    m  = x.shape[-1]
    zi = torch.cat((x, y), dim=-1)
    
    zk = None
    if xk is not None and yk is not None:
        if xk.ndim > 3:
            xk = xk.reshape(xk.shape[0], xk.shape[1], -1)
        zk = torch.cat((xk, yk), dim=-1)
    return x, zi, zk, m


# ---------------------------------------------------------------------------
# Step 2 — SVD local orthonormal basis
# ---------------------------------------------------------------------------

def _get_projection(args, x, xk):
    x_c = x.transpose(-2, -1)          # (D, b)
    x_c_mean = None

    if args.cems_method == 1:
        x_c_mean = torch.mean(x_c, -1)           # (D,)
        x_c = x_c - x_c_mean.unsqueeze(-1)  # singly-centred
    else:
        assert xk is not None
        xk_t = xk.transpose(-1, -2)
        x = x.unsqueeze(-1)
        x_c  = xk_t - x

    svd_input  = (x_c - x_c_mean.unsqueeze(-1) if x_c_mean is not None else x_c)
    svd_kwargs = {"driver": "gesvd"} if svd_input.is_cuda else {}
    basis, _, _ = torch.linalg.svd(svd_input, full_matrices=False, **svd_kwargs)

    u = basis.transpose(-2, -1) @ x_c  # (K, b)
    u_prev = u.transpose(-2, -1)             # (b, K)

    if args.cems_method == 1:
        u_t = u.transpose(-1, -2)                        # (b, K)
        u = (u_t.unsqueeze(1) - u_t).transpose(-1, -2) # (b, K, b)
        n = x.shape[0]
        mask = ~torch.eye(n, dtype=torch.bool, device=x.device)
        u = -u.transpose(-1, -2)[mask].reshape((u.shape[0], u.shape[2] - 1, u.shape[1])).transpose(-1, -2)  # (b, K, b-1)
    elif args.cems_method == 2:
        u = u.unsqueeze(0)

    return basis, u, u_prev, x_c_mean


# ---------------------------------------------------------------------------
# Ridge regression solver
# ---------------------------------------------------------------------------

def _solve_ridge(a, b, lam=1.0):
    n = a.shape[-1]
    eye = torch.eye(n, device=a.device, dtype=a.dtype)
    a_t = a.transpose(-2, -1)
    a_reg = a_t @ a + lam * eye
    return torch.linalg.inv(a_reg) @ a_t @ b


# ---------------------------------------------------------------------------
# Steps 3-4 — local gradient and Hessian via ridge regression
# ---------------------------------------------------------------------------

def _estimate_grad_hessian(args, x, xk, d):
    tidx = torch.triu_indices(d, d, device=x.device)
    ones_mult = torch.ones((d, d), device=x.device)
    ones_mult.fill_diagonal_(0.5)

    basis, u, u_prev, x_mean = _get_projection(args, x, xk)

    u_d = u[:, :d] # (b, d, b-1)
    f = u[:, d:].transpose(-2, -1) # (b, b-1, n_normal)

    if args.use_hessian:
        uu = torch.einsum('bki,bkj->bkij', u_d.transpose(-2, -1), u_d.transpose(-2, -1),)
        uu   = uu * ones_mult
        uu   = uu[:, :, tidx[0], tidx[1]].transpose(-2, -1)
        psi  = torch.cat((u_d, uu), dim=1).transpose(-2, -1)
    else:
        psi = u_d.transpose(-2, -1)  # CEMS-L: linear only

    lam    = torch.linalg.norm(psi, dim=(-1, -2)).mean()
    b_coef = _solve_ridge(psi, f, lam=lam).transpose(-2, -1)  # (b, n_normal, d or d+n_quad)

    gradient = b_coef[..., :d]
    hessian  = torch.zeros(
        (u.shape[0], b_coef.shape[1], d, d), dtype=b_coef.dtype, device=b_coef.device
    )
    if args.use_hessian:
        hessian[..., tidx[0], tidx[1]] = b_coef[..., d:]
        hessian[..., tidx[1], tidx[0]] = b_coef[..., d:]

    return basis, gradient, hessian, u_d, u_prev, x_mean


# ---------------------------------------------------------------------------
# Step 5 — sample in tangent bundle, project back to ambient space
# ---------------------------------------------------------------------------

def _sample_tangent(args, x, u_k_d, u_prev, x_mean, basis, grad, hess):
    d  = grad.shape[-1]
    nu = torch.distributions.Normal(0, args.sigma).sample(
        (x.shape[0], d, 1)
    ).to(x.device)

    f_nu = (grad @ nu).squeeze(-1)

    if args.use_hessian:
        nu_ex = nu.unsqueeze(1)
        f_nu  += 0.5 * (nu_ex.transpose(-1, -2) @ hess @ nu_ex).squeeze((-1, -2))

    x_new_local = torch.cat((nu.squeeze(-1), f_nu), dim=-1)

    if args.cems_method == 1:
        x_new_local += u_prev

    x_cems = (basis @ x_new_local.unsqueeze(-1)).squeeze(-1)
    x_cems += (x_mean if args.cems_method == 1 else x)
    return x_cems


# ---------------------------------------------------------------------------
# Public entry — get_batch_cems
# ---------------------------------------------------------------------------

def get_batch_cems(args, x, y, xk=None, yk=None, *, latent=True):
    """CEMS augmentation for one minibatch. Returns (x_new, y_new)."""
    x_shape, y_shape = x.shape, y.shape
    x_flat, zi, zk, m = _adjust_dims(x, y, xk, yk)

    d = args.id
    if latent:
        _d = twonn_inhouse(zi.detach().cpu().numpy())
        d = max(1, int(round(_d))) if math.isfinite(_d) and _d >= 1 else args.id

    if d < 1 or d >= zi.shape[-1] or d >= zi.shape[-2]:
        d = min(args.id, zi.shape[-1] - 1, zi.shape[-2] - 1)

    basis, grad, hess, u_k_d, u_prev, x_mean = _estimate_grad_hessian(args, zi, zk, d)
    z_sampled = _sample_tangent(args, zi, u_k_d, u_prev, x_mean, basis, grad, hess)

    x_new = z_sampled[..., :m].reshape(x_shape)
    y_new = z_sampled[..., m:].reshape(y_shape)
    return x_new, y_new


print("CEMS functions defined.")

## 11. Neighbourhood Utilities & TwoNN
precompute_knn for anchor-based CEMS; in-house TwoNN for global ID estimation.

> **Note — augmented twins as neighbours:** With 2× flip augmentation (identity + hflip_vflip), `precompute_knn` operates on a N_train×latent_dim matrix. The 2 orientations of any given image are genuinely close in feature space and share identical Y values, so they frequently end up as each other's nearest neighbours. This is intentional and correct — they are valid CEMS neighbourhood members.

In [ ]:
def _next_power_of_2(n):
    if n < 1:
        return 1
    return 1 << (n - 1).bit_length()


def compute_neigh_size(d):
    """neigh_size = next_power_of_2(d + d*(d+1)/2 + 1)."""
    base = d + d * (d + 1) // 2
    return _next_power_of_2(base + 1)


def precompute_knn(X, Y, neigh_size, compute_device="cpu"):
    """Pre-sort training points by Euclidean distance in joint [X, Y] space.

    Returns int64 array (N, neigh_size); col 0 = self-index.
    """
    XY = np.concatenate([X, Y], axis=1).astype(np.float32)
    t  = torch.tensor(XY, dtype=torch.float32)
    if compute_device in ("cuda", "mps"):
        try:
            t = t.to(compute_device)
        except Exception:
            pass  # fall back to CPU silently

    dists = torch.cdist(t, t, p=2, compute_mode="donot_use_mm_for_euclid_dist")
    dists.fill_diagonal_(-float('inf'))  # self sorts to col 0
    sorted_idx = torch.argsort(dists, dim=1).cpu().numpy()
    return sorted_idx[:, :neigh_size].astype(np.int64)


print("Neighbourhood utilities defined.")

## 12. Loss, Metrics & Dataset
Weighted SmoothL1, weighted global R², per-target RMSE, and a minimal PyTorch Dataset.

In [ ]:
WEIGHT_VECTOR = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
_LOSS_WEIGHTS = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)


def _weighted_smooth_l1(pred, target, weights, beta=1.0):
    """Per-target weighted SmoothL1, averaged over the batch."""
    loss_per = nn.functional.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss_per * weights.to(pred.device)).mean()


def weighted_global_r2(y_true, y_pred):
    """Kaggle competition metric: weighted global R² across all (image, target) pairs."""
    w      = WEIGHT_VECTOR
    yt     = y_true.reshape(-1)
    yp     = y_pred.reshape(-1)
    ww     = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


class BiomassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


@torch.no_grad()
def eval_epoch(model, loader, eval_device):
    model.eval()
    total_loss, all_pred, all_true = 0.0, [], []
    for X, y in loader:
        X, y = X.to(eval_device), y.to(eval_device)
        pred = model(X)
        total_loss += _weighted_smooth_l1(pred, y, _LOSS_WEIGHTS).item() * len(X)
        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    r2   = weighted_global_r2(y_true, y_pred)
    rmse = rmse_per_target(y_true, y_pred)
    return total_loss / len(loader.dataset), r2, rmse, y_pred, y_true


print("Loss, metrics, dataset, eval_epoch defined.")

## 13. Training — ERM + CEMS Augmentation
Single-phase training on the full BiomassModel (encoder + head jointly).

**Setup (once before training):**
1. Build `image_id_to_indices` from `image_group_ids[train_idx]` — maps each original image group to its variant indices in the training split.
2. Estimate `d_z` via TwoNN on `[X_train, Y_train_scaled]`; derive `neigh_size`.
3. Precompute kNN index on the same joint space.
4. Calibrate σ: if `sigma_auto=True`, compute median nearest-neighbour distance in z-space (optionally deduplicating to one variant per image via `sigma_dedup`), then `sigma = median_nn_dist × sigma_fraction`. Otherwise use `cfg.sigma` directly.

**Each epoch — orientation-aware without-replacement anchor sampling:**
- Shuffle the list of unique training image group IDs.
- Scan through them, skipping any already in `exhausted_images`.
- Collect the next B non-exhausted group IDs as anchors; add them immediately to `exhausted_images`.
- Pick one random variant index per anchor group.
- Fetch kNN neighbourhoods, run `get_batch_cems` in raw 384-d space, forward the B augmented samples through the full model.
- If `mix_real=True`: also forward the B real anchors and combine `loss = loss_real + λ·loss_aug`. If `False`: train on synthetic only.
- Repeat until the pool is empty; log `steps_this_epoch` after epoch 1.

In [ ]:
# ── Orientation group mapping ─────────────────────────────
train_group_ids = image_group_ids[train_idx]
unique_groups   = np.unique(train_group_ids)
N_unique_train  = len(unique_groups)
image_id_to_indices = {
    int(g): np.where(train_group_ids == g)[0].tolist()
    for g in unique_groups
}
print(f"N_train={len(X_train)}  N_unique_train={N_unique_train}  "
      f"variants/image={len(X_train) / N_unique_train:.1f}")

# ── CEMS args (sigma filled in after calibration below) ───
cems_args = SimpleNamespace(
    sigma=cfg.sigma,
    cems_method=cfg.cems_method,
    id=None,
    use_hessian=cfg.use_hessian,
)

# ── d_z estimation ────────────────────────────────────────
print("Estimating intrinsic dimension of [X_train, Y_train_scaled]...")
d_z_raw = twonn_inhouse(np.concatenate([X_train, Y_train_scaled], axis=1))
d_z_int = max(1, int(round(d_z_raw))) if math.isfinite(d_z_raw) and d_z_raw >= 1 else 5
neigh_size = compute_neigh_size(d_z_int)
cems_args.id = d_z_int
print(f"d_z={d_z_raw:.2f}  d_z_int={d_z_int}  neigh_size={neigh_size}")

# ── kNN on joint [X_train, Y_train_scaled] ────────────────
knn = precompute_knn(X_train, Y_train_scaled, neigh_size, compute_device=device)
print(f"kNN index: {knn.shape}  (col 0 = self)")

# ── Device tensors ────────────────────────────────────────
X_t     = torch.tensor(X_train,        dtype=torch.float32, device=device)
Y_raw_t = torch.tensor(Y_train_raw,    dtype=torch.float32, device=device)
Y_sc_t  = torch.tensor(Y_train_scaled, dtype=torch.float32, device=device)
samples_idx = np.arange(len(X_train))

# ── σ calibration ─────────────────────────────────────────
z_full = torch.cat([X_t, Y_sc_t], dim=1)   # (N_train, 389)
if cfg.sigma_auto:
    if cfg.sigma_dedup:
        rep_idx     = np.array([image_id_to_indices[int(g)][0] for g in unique_groups])
        z_for_sigma = z_full[rep_idx]        # (N_unique_train, 389) — one variant per image
    else:
        z_for_sigma = z_full                 # (N_train, 389)
    with torch.no_grad():
        nn_dists       = torch.cdist(z_for_sigma, z_for_sigma).topk(2, largest=False).values[:, 1]
        median_nn_dist = torch.median(nn_dists).item()
    sigma = median_nn_dist * cfg.sigma_fraction
    print(f"σ auto (sigma_dedup={cfg.sigma_dedup}): "
          f"median_nn_dist={median_nn_dist:.4f}  sigma_fraction={cfg.sigma_fraction}  → sigma={sigma:.6f}")
else:
    sigma = cfg.sigma
    print(f"σ fixed: {sigma}")
cems_args.sigma = sigma

# ── Val loader ────────────────────────────────────────────
val_ds     = BiomassDataset(X_val, Y_val_raw)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

print(f"\n≈ steps/epoch = ceil({N_unique_train} / {cfg.batch_size}) = {math.ceil(N_unique_train / cfg.batch_size)}")
print(f"input_sampling={cfg.input_sampling}  manifold_sampling={cfg.manifold_sampling}")
print(f"mix_real={cfg.mix_real}  lambda_aug={cfg.lambda_aug}  neigh_type={cfg.neigh_type}")

# ── Model, optimizer, scheduler ───────────────────────────
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
model     = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=cfg.lr / 100
)

history          = {k: [] for k in ["train_loss", "val_loss", "val_r2", "val_rmse"]}
best_val_r2      = -float("inf")
best_model_state = None
best_epoch       = 0

print("\n" + "=" * 62)
print("Training: ERM + CEMS augmentation (orientation-aware sampling)")
print("=" * 62)

for epoch in range(1, cfg.epochs + 1):
    model.train()

    shuffled_groups  = unique_groups[np.random.permutation(N_unique_train)]
    exhausted        = set()
    ep_loss          = 0.0
    ep_real_loss     = 0.0   # loss on real anchors (input_sampling + mix_real only)
    ep_aug_loss      = 0.0   # loss on CEMS-augmented samples (input_sampling only)
    n_seen           = 0
    steps_this_epoch = 0
    g_ptr            = 0

    while True:
        # ── Collect next B non-exhausted anchor group IDs ─
        anchor_gids = []
        while len(anchor_gids) < cfg.batch_size and g_ptr < N_unique_train:
            gid = int(shuffled_groups[g_ptr])
            g_ptr += 1
            if gid not in exhausted:
                anchor_gids.append(gid)
        if not anchor_gids:
            break
        B = len(anchor_gids)

        exhausted.update(anchor_gids)

        # ── One random variant index per anchor group ──────
        idx_1 = np.array([
            np.random.choice(image_id_to_indices[gid]) for gid in anchor_gids
        ])

        # ── Neighbour indices (skip col-0 self) ───────────
        if cfg.neigh_type == "knn":
            idx_2 = knn[idx_1][:, 1:]
        else:
            idx_2 = np.array([
                np.random.choice(samples_idx, size=neigh_size - 1, replace=False)
                for _ in range(B)
            ])

        X_i   = X_t[idx_1]
        Y_i_s = Y_sc_t[idx_1]
        Y_i_r = Y_raw_t[idx_1]
        X_k   = X_t[idx_2]
        Y_k_s = Y_sc_t[idx_2]

        # ── Augmentation + forward ────────────────────────
        if cfg.input_sampling:
            X_mix, Y_mix_s = get_batch_cems(cems_args, X_i, Y_i_s, X_k, Y_k_s)
            Y_mix_r = torch.tensor(
                y_scaler.inverse_transform(Y_mix_s.detach().cpu().numpy().astype(np.float32)),
                dtype=torch.float32, device=device,
            )
            pred_aug = model(X_mix)
            loss_aug = _weighted_smooth_l1(pred_aug, Y_mix_r, _LOSS_WEIGHTS)
            ep_aug_loss += loss_aug.item() * B

            if cfg.mix_real:
                pred_real = model(X_i)
                loss_real = _weighted_smooth_l1(pred_real, Y_i_r, _LOSS_WEIGHTS)
                ep_real_loss += loss_real.item() * B
                loss = loss_real + cfg.lambda_aug * loss_aug
            else:
                loss = loss_aug

        elif cfg.manifold_sampling:
            pred, Y_mix_s = model.forward_cems(cems_args, X_i, Y_i_s)
            Y_mix_r = torch.tensor(
                y_scaler.inverse_transform(Y_mix_s.detach().cpu().numpy().astype(np.float32)),
                dtype=torch.float32, device=device,
            )
            loss = _weighted_smooth_l1(pred, Y_mix_r, _LOSS_WEIGHTS)

        else:
            pred = model(X_i)
            loss = _weighted_smooth_l1(pred, Y_i_r, _LOSS_WEIGHTS)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ep_loss += loss.item() * B
        n_seen  += B
        steps_this_epoch += 1

    scheduler.step()
    tr_loss = ep_loss      / n_seen
    tr_real = ep_real_loss / n_seen   # 0 when not applicable
    tr_aug  = ep_aug_loss  / n_seen   # 0 when not applicable

    val_loss, val_r2, val_rmse, _, _ = eval_epoch(model, val_loader, device)

    if val_r2 > best_val_r2:
        best_val_r2      = val_r2
        best_epoch       = epoch
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    history["val_r2"].append(val_r2)
    history["val_rmse"].append(val_rmse)

    if epoch == 1:
        print(f"  steps/epoch (confirmed after epoch 1): {steps_this_epoch}  "
              f"(N_unique_train // batch_size = {N_unique_train // cfg.batch_size})")

    if epoch % 5 == 0 or epoch == 1:
        rmse_str  = "  ".join(f"{t.split('_')[1]}:{v:.2f}" for t, v in val_rmse.items())
        loss_str  = f"loss={tr_loss:.4f}"
        if cfg.input_sampling:
            loss_str += f"  real={tr_real:.4f}  aug={tr_aug:.4f}"
        print(
            f"  ep {epoch:3d}  {loss_str}  val_loss={val_loss:.4f}  "
            f"val_R²={val_r2:.4f}  [{rmse_str}]"
        )

assert best_model_state is not None
model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
model.eval()
print(f"\nBest val R² = {best_val_r2:.4f} at epoch {best_epoch}")

## 14. Training Trajectory Plot
Val R² and loss over all epochs. Vertical dashed line marks the best-val-R² epoch.

In [ ]:
import matplotlib.pyplot as plt

epochs_axis = list(range(1, len(history["val_r2"]) + 1))
best_r2_epoch = int(np.argmax(history["val_r2"])) + 1
best_r2_val   = max(history["val_r2"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(epochs_axis, history["val_r2"], color="steelblue", linewidth=1.5,
         label=f"Val R²  (best={best_r2_val:.4f} @ ep {best_r2_epoch})")
ax1.axvline(best_r2_epoch, color="black", linestyle="--", linewidth=1.2, alpha=0.7,
            label=f"Best epoch {best_r2_epoch}")
ax1.set_ylabel("Val weighted R²")
ax1.set_title("Training Trajectory")
ax1.legend(loc="lower right", fontsize=9)

ax2.plot(epochs_axis, history["train_loss"], color="darkorange", linewidth=1.5, label="Train loss")
ax2.plot(epochs_axis, history["val_loss"],   color="steelblue",  linewidth=1.5, label="Val loss")
ax2.axvline(best_r2_epoch, color="black", linestyle="--", linewidth=1.2, alpha=0.7)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Weighted SmoothL1 Loss")
ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nBest val R² = {best_r2_val:.4f} at epoch {best_r2_epoch}")

## 15. Final Validation Metrics
Evaluate best checkpoint; flag if weighted R² < 0.75 (possible extraction mismatch).

In [ ]:
model.eval()
with torch.no_grad():
    val_preds = model(
        torch.tensor(X_val, dtype=torch.float32, device=device)
    ).cpu().numpy()

val_r2_best   = weighted_global_r2(Y_val_raw, val_preds)
val_rmse_best = rmse_per_target(Y_val_raw, val_preds)

print("=" * 55)
print(f"Final val weighted R²: {val_r2_best:.4f}  (best epoch {best_epoch})")
print("=" * 55)
print("RMSE per target:")
for t, v in val_rmse_best.items():
    print(f"  {t:<18}: {v:.4f}")
print("=" * 55)

if val_r2_best < 0.75:
    print(
        "\n  *** WARNING: val R² = {:.4f} is below expected baseline (~0.80).".format(val_r2_best)
    )
else:
    print(f"\n  R² = {val_r2_best:.4f} — consistent with baseline. OK.")

## 16. Test Inference
Load best checkpoint, encode test features through encoder+head, inverse-scale predictions.

In [ ]:
model.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

with torch.no_grad():
    test_preds = model(X_test_t).cpu().numpy()

test_preds = np.clip(test_preds, 0.0, None)

print(f"test_preds shape: {test_preds.shape}")
print(f"test_preds range: [{test_preds.min():.2f}, {test_preds.max():.2f}]")
print("Sample predictions (first 3 test images):")
for i in range(min(3, len(test_image_ids))):
    print(f"  {test_image_ids[i]}: {dict(zip(TARGETS, test_preds[i].round(2)))}")

## 17. Format Predictions as Submission CSV
Map (image_id, 5 targets) → long-format rows matching test.csv sample_id keys.

In [17]:
def prepare_submission(test_csv_path, predictions, image_ids):
    """Returns long-format DataFrame with columns [sample_id, target]."""
    df_t = pd.read_csv(test_csv_path)

    pred_dict = {
        img_id: {col: float(val) for col, val in zip(TARGETS, pred_row)}
        for img_id, pred_row in zip(image_ids, predictions)
    }

    def _get_pred(row):
        img_id      = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        val = pred_dict.get(img_id, {}).get(target_name, 0.0)
        return max(0.0, val)

    df_t["target"] = df_t.apply(_get_pred, axis=1)
    return df_t[["sample_id", "target"]]


submission = prepare_submission(TEST_CSV, test_preds, test_image_ids)

out_path = os.path.join(cfg.output_dir, "submission.csv")
submission.to_csv(out_path, index=False)

print(f"Submission saved to: {out_path}")
print(f"Rows: {len(submission)}")
print(submission.head(10))
print("\nTarget value stats:")
print(submission["target"].describe().round(3))
print("\nWorking dir:", os.listdir(cfg.output_dir))

Submission saved to: /kaggle/working/submission.csv
Rows: 5
                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.996477
1    ID1001187975__Dry_Dead_g  11.537954
2   ID1001187975__Dry_Green_g  28.398699
3   ID1001187975__Dry_Total_g  52.660595
4         ID1001187975__GDM_g  33.906761

Target value stats:
count     5.000
mean     25.500
std      20.076
min       0.996
25%      11.538
50%      28.399
75%      33.907
max      52.661
Name: target, dtype: float64

Working dir: ['two_stage_trajectory.png', '.virtual_documents', 'submission.csv']
